In [25]:
#must pip install pygame and music21


import tkinter as tk
from tkinter import messagebox
from qiskit import QuantumCircuit 
from qiskit_aer import Aer
from datetime import datetime
from music21 import note, stream, duration, tempo, instrument 
import os
import time
import sys 


try:
    import pygame
    pygame.mixer.init() 
except ImportError:
    pygame = None
    print("Warning: pygame not found. Install with 'pip install pygame' to enable playback.")
except pygame.error as e:
    pygame = None
    print(f"Warning: Pygame mixer failed to initialize ({e}). Playback disabled.")




def birthday_to_angles(date_str):
    """Converts a date string (YYYYMMDD) into a list of quantum rotation angles (0 to pi)."""
    digits = [int(d) for d in date_str]
    angles = [(d / 9.0) * 3.1415926535 for d in digits]
    return angles

def build_quantum_circuit(angles):
    """Builds a quantum circuit using the input angles."""
    num_qubits = len(angles)
    qc = QuantumCircuit(num_qubits, num_qubits)
    qc.h(range(num_qubits)) 
    for i, angle in enumerate(angles):
        qc.rx(angle, i) 
    for i in range(num_qubits - 1):
        qc.cx(i, i+1) 
    qc.measure(range(num_qubits), range(num_qubits))
    return qc

def get_quantum_results(qc, shots=4096):
    """Runs the circuit on a Qiskit simulator and returns the counts (probabilities)."""
    simulator = Aer.get_backend('qasm_simulator')
    job = simulator.run(qc, shots=shots) 
    result = job.result()
    counts = result.get_counts(qc)
    return counts

def map_results_to_music(counts, base_note='C4', scale_notes=12):
    """Maps measurement outcomes to musical notes and durations."""
    music_stream = stream.Stream()
    base_midi = note.Note(base_note).pitch.midi 
    sorted_counts = sorted(counts.items(), key=lambda item: item[1], reverse=True)
    
    for outcome, count in sorted_counts:
        probability = count / sum(counts.values())
        pitch_shift = int(outcome, 2) % scale_notes
        midi_pitch = base_midi + pitch_shift
        note_duration = duration.Duration(probability * 4.0) 
        new_note = note.Note(midi_pitch, duration=note_duration)
        music_stream.append(new_note)
        
    return music_stream



class QuantumMusicApp:
    
    TARGET_TEMPO = 2 

    def __init__(self, master):
        self.master = master
        master.title("Quantum Birthday Music Generator")
        master.config(padx=20, pady=20)
        self.midi_file_path = None 

        
        tk.Label(master, text="Enter Birthday (YYYYMMDD):", font=('Arial', 12, 'bold')).grid(row=0, column=0, pady=5, sticky='w')
        self.birthday_entry = tk.Entry(master, width=10, font=('Arial', 12))
        self.birthday_entry.insert(0, "20010101")
        self.birthday_entry.grid(row=0, column=1, pady=5, sticky='ew')
        
        
        tk.Label(master, text="Change Instrument (1-128):", font=('Arial', 12, 'bold')).grid(row=1, column=0, pady=5, sticky='w')
        self.instrument_entry = tk.Entry(master, width=10, font=('Arial', 12))
        self.instrument_entry.insert(0, "11") # Default to Violin (41)
        self.instrument_entry.grid(row=1, column=1, pady=5, sticky='ew')
        
        
        self.generate_button = tk.Button(master, text="GENERATE QUANTUM SONG 🎶", 
                                         command=self.generate_song, 
                                         bg='#6a0dad', fg='white', font=('Arial', 14, 'bold'), relief=tk.RAISED)
        self.generate_button.grid(row=2, column=0, columnspan=2, pady=(20, 10), ipadx=10, ipady=5, sticky='ew')

        
        self.play_button = tk.Button(master, text="▶️ PLAY SONG", 
                                     command=self.play_song, 
                                     bg='#00aaff', fg='white', font=('Arial', 12), relief=tk.RAISED, state=tk.DISABLED)
        self.play_button.grid(row=3, column=0, padx=5, pady=5, sticky='ew')

        self.stop_button = tk.Button(master, text="⏹ STOP", 
                                     command=self.stop_song, 
                                     bg='#ff0000', fg='white', font=('Arial', 12), relief=tk.RAISED, state=tk.DISABLED)
        self.stop_button.grid(row=3, column=1, padx=5, pady=5, sticky='ew')
        
        
        self.status_var = tk.StringVar()
        self.status_var.set("Ready. Input an 8-digit birthday to begin.")
        self.status_label = tk.Label(master, textvariable=self.status_var, 
                                     font=('Arial', 10), fg='darkgreen', wraplength=400, justify=tk.LEFT)
        self.status_label.grid(row=4, column=0, columnspan=2, pady=10, sticky='w')


    def generate_song(self):
        """Handles user input, runs the quantum logic, and saves the MIDI file."""
        birthday_str = self.birthday_entry.get()
        instrument_str = self.instrument_entry.get()
        
        
        if len(birthday_str) != 8 or not birthday_str.isdigit():
            messagebox.showerror("Invalid Input", "Please enter a birthday in the format YYYYMMDD.")
            return
            
        try:
            instrument_number = int(instrument_str)
            if not 1 <= instrument_number <= 128:
                raise ValueError
        except ValueError:
            messagebox.showerror("Invalid Instrument", "Please enter an instrument number between 1 and 128.")
            return

        self.status_var.set("Status: Building Quantum Circuit...")
        self.master.update()
        
        try:
            angles = birthday_to_angles(birthday_str)
            qc = build_quantum_circuit(angles)
            counts = get_quantum_results(qc)
            note_stream = map_results_to_music(counts)

            
            chosen_instrument = instrument.instrumentFromMidiProgram(instrument_number - 1)
            music_part = stream.Part()
            music_part.insert(0, chosen_instrument)
            music_part.append(note_stream)

            final_score = stream.Score()
            music_tempo = tempo.MetronomeMark(number=self.TARGET_TEMPO) 
            final_score.insert(0, music_tempo)
            final_score.insert(0, music_part)
            
            
            midi_filename = f"quantum_song_{birthday_str}_inst{instrument_number}.mid"
            final_score.write('midi', fp=midi_filename)
            self.midi_file_path = os.path.abspath(midi_filename)

            
            if pygame:
                self.play_button.config(state=tk.NORMAL)
                self.stop_button.config(state=tk.NORMAL)

            
            success_message = (f"✅ **Success!** Song saved as:\n"
                               f"**{os.path.basename(midi_filename)}**\n"
                               f"Ready to play!")
            self.status_var.set(success_message)
            messagebox.showinfo("Generation Complete", f"Song saved and ready to play.")

        except Exception as e:
            error_message = f"An unexpected error occurred:\n{e}"
            self.status_var.set(f"Error: {e}")
            messagebox.showerror("Generation Error", error_message)

    
    def play_song(self):
        """Plays the generated MIDI file using pygame."""
        if not pygame:
            messagebox.showerror("Playback Error", "Pygame library is not available.")
            return

        if self.midi_file_path and os.path.exists(self.midi_file_path):
            try:
                self.stop_song()
                time.sleep(0.1) 
                
                pygame.mixer.music.load(self.midi_file_path)
                pygame.mixer.music.play()
                self.status_var.set(f"▶️ Now Playing: {os.path.basename(self.midi_file_path)}")
            except pygame.error as e:
                messagebox.showerror("Playback Error", f"Could not play MIDI file:\n{e}")
                self.status_var.set("Playback Error.")
        else:
            messagebox.showwarning("File Missing", "Please generate a song first.")

    def stop_song(self):
        """Stops the currently playing MIDI file."""
        if pygame and pygame.mixer.music.get_busy():
            pygame.mixer.music.stop()
            self.status_var.set("⏹ Playback stopped.")
        elif pygame:
            self.status_var.set("Ready.")

    def on_closing(self):
        """Stops the music and cleanly exits the application."""
        if pygame and pygame.mixer.music.get_busy():
            pygame.mixer.music.stop()
        
        self.master.destroy()
        
        sys.exit() 


if __name__ == "__main__":
    root = tk.Tk()
    app = QuantumMusicApp(root)
    
    root.protocol("WM_DELETE_WINDOW", app.on_closing)
    
    root.mainloop()

SystemExit: 